# HMM Information-Gap Correction

This notebook computes only the corrected same-sequence information gap

E[ KL( p_long(x_next | long history) || p_L(x_next | last L observations) ) ]

for the already completed HMM experiment.

No Transformer training, checkpoint loading, or probing is performed.

Optimizations:
- HMM sampling on CUDA;
- Bayesian filtering on CUDA;
- no target-token sampling;
- no redundant empirical CE-difference calculation;
- streamed batches to keep memory low.

In [ ]:
import json, math, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "CUDA GPU required for this optimized notebook."

BASE=Path.cwd()/"Markov_Context_Paper"
OLD_OUT=BASE/"HMM_Latent_State_Extension"
OUT=BASE/"HMM_Oracle_Gap_Fix_GPU"
CSV_DIR=OUT/"csv"
FIG_DIR=OUT/"figures"
for p in [OUT,CSV_DIR,FIG_DIR]:
    p.mkdir(parents=True,exist_ok=True)

V=8
KZ=8
SEEDS=[0,1,2]
CONTEXT_LENGTHS=[2,4,8,16,32]
TRANSITION_STICKINESS=5.0
EMISSION_CONCENTRATION=0.55
LONG_HISTORY=256

# 100k per seed matches the intended robust post-processing check.
N_SAMPLES=100_000
BATCH_SIZE=8192

print("Device:",DEVICE)
print("GPU:",torch.cuda.get_device_name(0))
print("Samples per seed:",N_SAMPLES)
print("Batch size:",BATCH_SIZE)


In [ ]:
def make_hmm(seed,V=V,K=KZ):
    rng=np.random.default_rng(seed+91337)

    raw_T=rng.gamma(shape=1.0,scale=1.0,size=(K,K))
    raw_T += TRANSITION_STICKINESS*np.eye(K)
    T=raw_T/raw_T.sum(axis=1,keepdims=True)

    raw_E=rng.gamma(shape=EMISSION_CONCENTRATION,scale=1.0,size=(K,V))
    E=raw_E/raw_E.sum(axis=1,keepdims=True)

    pi=np.full(K,1.0/K)
    for _ in range(100000):
        new=pi@T
        if np.max(np.abs(new-pi))<1e-14:
            pi=new
            break
        pi=new
    pi=pi/pi.sum()

    return (
        torch.tensor(T,dtype=torch.float64,device=DEVICE),
        torch.tensor(E,dtype=torch.float64,device=DEVICE),
        torch.tensor(pi,dtype=torch.float64,device=DEVICE),
    )

def categorical_rows(probs, generator):
    # probs: [B,C], normalized
    u=torch.rand((probs.shape[0],1),device=probs.device,generator=generator)
    cdf=torch.cumsum(probs,dim=-1)
    return (u>cdf).sum(dim=-1).long()

@torch.no_grad()
def sample_observation_histories(T,E,pi,seq_len,batch_size,seed):
    g=torch.Generator(device=DEVICE)
    g.manual_seed(seed)

    # Initial latent state from stationary prior.
    p0=pi.float()[None,:].expand(batch_size,-1)
    z=categorical_rows(p0,g)

    xs=[]
    x=categorical_rows(E[z].float(),g)
    xs.append(x)

    for _ in range(1,seq_len):
        z=categorical_rows(T[z].float(),g)
        x=categorical_rows(E[z].float(),g)
        xs.append(x)

    return torch.stack(xs,dim=1)

@torch.no_grad()
def predictive_distribution(context,T,E,pi):
    B,L=context.shape
    post=pi[None,:].expand(B,-1).clone()

    for t in range(L):
        obs=context[:,t]
        likelihood=E[:,obs].T
        post.mul_(likelihood)
        post.div_(post.sum(dim=1,keepdim=True).clamp_min(1e-300))

        if t<L-1:
            post=post@T

    latent_next=post@T
    pred=latent_next@E
    pred.div_(pred.sum(dim=1,keepdim=True).clamp_min(1e-300))
    return pred


## Same-history KL estimator

The finite-context predictor and long-history predictor are evaluated for the exact same sampled observation history.

The primary estimator is non-negative up to floating-point roundoff.

In [ ]:
rows=[]

torch.cuda.synchronize()
overall_start=time.time()

for seed in SEEDS:
    seed_start=time.time()
    print(f"\n=== seed={seed} ===")
    T,E,pi=make_hmm(seed)

    sums={L:0.0 for L in CONTEXT_LENGTHS}
    sums_sq={L:0.0 for L in CONTEXT_LENGTHS}
    total_n=0

    n_batches=math.ceil(N_SAMPLES/BATCH_SIZE)

    for b in range(n_batches):
        n_this=min(BATCH_SIZE,N_SAMPLES-total_n)
        if n_this<=0:
            break

        history=sample_observation_histories(
            T,E,pi,
            seq_len=LONG_HISTORY,
            batch_size=n_this,
            seed=50_000_000+seed*100_000+b
        )

        p_long=predictive_distribution(history,T,E,pi)
        log_p_long=torch.log(p_long.clamp_min(1e-300))

        for L in CONTEXT_LENGTHS:
            p_L=predictive_distribution(history[:,-L:],T,E,pi)
            kl=(
                p_long*
                (log_p_long-torch.log(p_L.clamp_min(1e-300)))
            ).sum(dim=-1)

            sums[L] += float(kl.sum().item())
            sums_sq[L] += float((kl*kl).sum().item())

        total_n += n_this

        del history,p_long,log_p_long
        if (b+1)%4==0 or b+1==n_batches:
            torch.cuda.synchronize()
            print(f"  batch {b+1}/{n_batches} | samples={total_n}")

    for L in CONTEXT_LENGTHS:
        mean=sums[L]/total_n
        mean_sq=sums_sq[L]/total_n
        var=max(0.0,mean_sq-mean*mean)
        se=math.sqrt(var/total_n)

        rows.append({
            "seed":seed,
            "L":L,
            "n_samples":total_n,
            "information_gap_KL_nats":mean,
            "mc_standard_error":se,
        })

    torch.cuda.synchronize()
    print(f"seed {seed} time: {(time.time()-seed_start)/60:.2f} min")

df=pd.DataFrame(rows)
df.to_csv(CSV_DIR/"HMM_information_gap_same_sequence_GPU.csv",index=False)

torch.cuda.synchronize()
print(f"\nTotal time: {(time.time()-overall_start)/60:.2f} min")
display(df)


In [ ]:
summary=df.groupby("L",as_index=False).agg(
    information_gap_mean=("information_gap_KL_nats","mean"),
    information_gap_sd_across_seeds=("information_gap_KL_nats","std"),
    mean_mc_standard_error=("mc_standard_error","mean"),
    n_seeds=("seed","nunique"),
)

summary.to_csv(
    CSV_DIR/"HMM_information_gap_same_sequence_GPU_summary.csv",
    index=False
)

assert (df["information_gap_KL_nats"]>=-1e-12).all()
assert np.isfinite(df.select_dtypes(include=[np.number]).to_numpy()).all()

display(summary)
print("Minimum KL gap:",df["information_gap_KL_nats"].min())
print("Numerical audit: PASS")


In [ ]:
fig,ax=plt.subplots(figsize=(7.5,4.8))

ax.plot(
    summary["L"],
    summary["information_gap_mean"],
    marker="o",
    label="Finite-context information gap (same-history KL)"
)

old_summary_path=OLD_OUT/"csv"/"HMM_native_oracle_summary.csv"
if old_summary_path.exists():
    old=pd.read_csv(old_summary_path)
    ax.plot(
        old["L"],
        old["model_excess_mean"],
        marker="o",
        label="Model excess over H_L"
    )

ax.axhline(0.0,linewidth=1)
ax.set_xscale("log",base=2)
ax.set_yscale("symlog",linthresh=1e-7)
ax.set_xlabel("Context length L")
ax.set_ylabel("Gap (nats)")
ax.set_title("HMM information limitation vs model/optimization gap")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_gap_decomposition_CORRECTED_GPU.png",dpi=300)
plt.show()


In [ ]:
audit={
    "status":"PASS",
    "method":"same-history expected KL between long-history and finite-context exact Bayesian predictive distributions",
    "device":str(DEVICE),
    "seeds":SEEDS,
    "context_lengths":CONTEXT_LENGTHS,
    "long_history":LONG_HISTORY,
    "samples_per_seed":N_SAMPLES,
    "all_KL_nonnegative_with_tolerance":bool(
        (df["information_gap_KL_nats"]>=-1e-12).all()
    ),
}

with open(OUT/"HMM_oracle_gap_fix_GPU_audit.json","w",encoding="utf-8") as f:
    json.dump(audit,f,indent=2)

zip_path=Path.cwd()/"HMM_Oracle_Gap_Fix_GPU_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f,f.relative_to(OUT.parent))

print(json.dumps(audit,indent=2))
print("Created:",zip_path)
